_Neural Data Science_

Lecturer: Dr. Jan Lause, Prof. Dr. Philipp Berens

Tutors: Jonas Beck, Fabio Seel, Julius Würzler

Summer term 2025

Student names: Luca Kosina, Sascha Mühlinghaus, Max Bardelang

LLM Disclaimer: ...

# Neural Data Science Project 01

## Working with Calcium data

In the following project you will recieve a data set, along with a set of questions. Use the methods that you have learned throughout this course to explore the data and to answer the questions. You are free to use tools, resources and libraries as you see fit. Use comments and markdown cells to document your thought process and to explain your reasoning. We encourage you to compare different algorithms or to implement state of the art solutions. The notebook should be self contained, although you may offload some functions to a `utils.py`. The notebook should be concluded with a final summary / conclusions section.

In [ ]:
# import packages here

import numpy as np
import pandas as pd
import jupyter_black

jupyter_black.load()

## Context


![image.png](attachment:image.png)

The data set that goes along with this notebook was recorded using in vivo 2-photon calcium imaging to measure the activity of genetically identified neurons in the visual cortex of mice performing a go/no-go visual change detection task. The data recordings stem from primary visual cortex and a GCaMP6f indicator was used. The data was recorded as follows.

![image-3.png](attachment:image-3.png)

The data consists of:
- the preprocessed activity traces (df/f)
- the stimulus metadata
- the ROI masks for each cell
- a maximum activity projection of the recorded area
- running speed
- table of stimulus epochs

You will only work with a drifting grating stimulus.

Since the experiments were performed in sequence the calcium recordings that you receive also contain some other stimulus modalities (see `data["stim_epoch_table"]`). You can ignore these sections of the time-series data during analysis. Not all the data provided has to be used, however it can be incorporated into your analysis.

In [ ]:
# load data
def load_data(path="."):
    def array2df(d, key, cols):
        d[key] = pd.DataFrame(d[key], columns=cols)

    data = np.load(path + "/dff_data_dsi.npz", allow_pickle=True)
    data = dict(data)
    array2df(
        data,
        "stim_table",
        ["frequency", "direction", "blank_sweep", "start", "end"],
    )
    array2df(data, "stim_epoch_table", ["start", "end", "stimulus"])

    return data


def print_info(data):
    data_iter = ((k, type(v), v.shape) for k, v in data.items())
    l = [f"[{k}] - {t}, - {s}" for k, t, s in data_iter]
    print("\n".join(l) + "\n")


data = load_data(path="../data/project-01")  # adjust the path as necessary

print("Overview of the data")
print_info(data)
print(data["dff"].shape)

## Question

**Is there spatial structure in the preferred orientation/direction/frequency?**

Implement all steps of the processing pipeline that are necessary to answer them. Think of:
1. Pre-processing
2. Spike inference
3. Tuning function fitting
4. Statistical testing.

It is sufficient to assess spatial structure visually. Additional insights and analyses will be positively factored into the overall grade.

# Implementation

We are following these steps to answer the questions:

We filter the data to have smoother signals by cutting out low frequencies to be able to detect spikes in high-frequency components.
We then calculate the threshold for spikes and detect positions of spikes in the filtered data.
Because we are interested in the specificity of neurons to stimuls characteristics we look at each trial of stimulus presentation and segment the reactions of neurons in each stimulus segment.
Since only certain trials show detect spikes we filter out trials for each neurons in which spikes occur to continue the analysis with these.
We then correct each stimulus reaction to the baseline activity of the neuron before the stimulus presentation.
We then fit tuning curves to the detected spikes after stimulus presentation and by permutation analysis check which neurons have certain orientation, direction and frequency preference.
Then using the mask we plot the entire filed of neurons with their respective preference to first check if we can visually see an organization of the neurons into columns.
We statistically analyze this by ...



## Pre-processing

We start by looking at what the the raw data looks like for one example neuron:

In [ ]:
# plot dff
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", palette="muted", font_scale=1.2)


def plot_dff(data, neuron=0):
    """
    Plot the dff data for a given neuron and stimulus epoch.

    Parameters:
    - data: The data dictionary containing dff data.
    - neuron: The index of the neuron to plot.
    - stim_epoch: The index of the stimulus epoch to plot.
    """
    dff = data["dff"]

    plt.figure(figsize=(10, 5))
    plt.plot(
        data["t"] / 60,
        dff[neuron],
        label=f"Neuron {neuron}",
    )
    plt.title("DFF Data")
    plt.xlabel("time (minutes)")
    plt.ylabel("DFF")
    plt.xlim(0, 65)
    plt.show()


# Example plot for one neuron
plot_dff(data, neuron=0)

To see more of the frequency structure let´s look at one trial of the data:

In [ ]:
data["stim_table"]

In [ ]:
# look at specific trial (segment of the data) for the example neuron
plt.figure(figsize=(10, 5))

# extract stimulus epoch for the example neuron
stim_epoch = data["stim_table"].iloc[0]  # First stimulus epoch
start, end = int(stim_epoch["start"]), int(stim_epoch["end"])

plt.plot(
    data["t"][start:end],
    data["dff"][0][start:end],
    label="Neuron 0",
)
plt.title("DFF Neuron 0, Trial 1")
plt.xlabel("Time (s)")
plt.ylabel("DFF")
plt.show()

We can see that there is a slight low frequency oscillation in the data which we now want to filter out.

Were utilizing a butterworth filter with parameters as in Yaksi, Friedrichs (2006):
- low-pass
- cutoff: 0.2 * frame_rate
- 4-pole

In [ ]:
from scipy.signal import butter, filtfilt


def butterworth_filter(data, cutoff, fs, order):
    """
    Apply a Butterworth filter to the data.

    Parameters:segment
    - data: The data to filter.
    - cutoff: The cutoff frequency of the filter.
    - fs: The sampling frequency of the data.
    - order: The order of the filter.

    Returns:
    - The filtered data.
    """
    b, a = butter(order, cutoff, btype="lowpass", analog=False, fs=fs)
    filtered_data = filtfilt(b, a, data)
    return filtered_data

After the filter is applied, the previous example trial looks like this

In [ ]:
plt.plot(
    data["t"][start:end],
    butterworth_filter(data["dff"][0][start:end], cutoff=0.2 * 30, fs=30, order=4),
);

Apply the filter to the dff data for one example neuron

In [ ]:
filtered_dff = butterworth_filter(data["dff"][0], cutoff=0.2 * 30, fs=30, order=4)
# Plot the filtered dff data
plt.figure(figsize=(10, 5))
plt.plot(data["dff"][0], label=f"Unfiltered Data", color="blue", alpha=0.5)
plt.plot(filtered_dff, label="Filtered DFF", color="orange", alpha=0.5)
plt.title("Filtered DFF Data Neuron 0")
plt.xlabel("Time")
plt.ylabel("DFF")
plt.legend()
plt.show()

A smoothed signal reduces noise while preserving essential features of neural activity. This will help in further analyses such as identifying peaks, calculating statistics, or visualizing the data more clearly.

Apply the filter to all neurons

In [ ]:
# apply filter to all neurons
filtered_dff_all = np.array(
    [butterworth_filter(d, cutoff=0.2 * 30, fs=30, order=4) for d in data["dff"]]
)

## Spike Inference

We can now continue with the analysis using the filtered data. First, we detect the spikes in the entire dataset for each neuron.

In [ ]:
def detect_spikes(
    x: np.ndarray, fs: float, N: int = 5, lockout: float = 1.0
) -> tuple[np.ndarray, np.ndarray, np.float64]:
    """Detect spikes in the signal x and compute a threshold.

    Parameters
    ----------

    x: np.array (n_samples, n_channels)
        The filtered signal from Task 1.

    fs: float
        the sampling rate (in Hz).

    N: int
        An arbitrary number with which you multiply with the standard deviation
        to set a threshold that controls your false positive rate. Default is 5
        but you should try changing it and see how it affects the results.

    lockout: float
        a window of 'refractory period', within which there's only one spike.
        Default is 1ms but you should also try changing it.


    Returns
    -------

    s: np.array, (n_spikes, )
        Spike location / index in the signal x.

    t: np.array, (n_spikes, )
        Spike time in ms. By convention the time of the zeroth sample is 0 ms.

    thrd: float
        Threshold = -N * sigma.


    Tips
    ----

    You can use scipy functions like find_peaks for the detection.
    Note: There are four channels in signal x.

    """
    # compute the robust s.d. and calculate the threshold
    sigma = np.median(np.abs(x - np.mean(x))) / 0.6745
    threshold = N * sigma

    # candidate positive peaks
    pos_peaks, _ = signal.find_peaks(x, distance=lockout)
    pos_peaks = pos_peaks[x[pos_peaks] >= threshold]

    # negative peaks are not used because they coocur too close to the positive peaks
    peaks = pos_peaks

    spike_times = peaks * (1000 / fs)  # convert to ms

    return peaks, spike_times, threshold

In [ ]:
# detect spikes in the filtered dff data for example neuron
spikes, spike_times, threshold = detect_spikes(filtered_dff, fs=30, N=3, lockout=1.0)

# plot the detected spikes
plt.figure(figsize=(10, 5))
plt.plot(filtered_dff, label="Filtered DFF", color="orange", alpha=0.5)
plt.scatter(
    spikes, filtered_dff[spikes], color="red", label="Detected Spikes", marker="x"
)
plt.title("Detected Spikes in Filtered DFF Data")
plt.xlabel("Time (samples)")
plt.ylabel("DFF")
plt.legend()
plt.show()

In [ ]:
# look at specific short segment of the data
plt.figure(figsize=(10, 5))
plt.plot(filtered_dff[0:5000], label="Filtered DFF", color="orange", alpha=0.5)

spikes, spike_times, threshold = detect_spikes(
    filtered_dff[0:5000], fs=30, N=3, lockout=1.0
)


# add spikes
plt.scatter(
    spikes,
    filtered_dff[spikes],
    color="red",
    label="Detected Spikes",
    marker="x",
)
plt.title("Detected Spikes in Filtered DFF Data")
plt.xlabel("Time (samples)")
plt.ylabel("DFF")
plt.legend()
plt.show()

Again we apply the spike detection to all neurons:

In [ ]:
# spike detection for all neurons
N = 2
spikes_all = []
spike_times_all = []
thresholds_all = []
for dff in filtered_dff_all:
    spikes, spike_times, threshold = detect_spikes(dff, fs=1000, N=N, lockout=1.0)
    spikes_all.append(spikes)
    spike_times_all.append(spike_times)
    thresholds_all.append(threshold)

## OOPSIE algorithm

In [ ]:
# d: reconstructed Spikes
import oopsi

dt = 0.0333  # sampling interval in seconds
ogb_d, ogb_Cz = oopsi.fast(filtered_dff, dt=dt, iter_max=6)

## Segment data by stimulus presentation and epochs

Since we want to analyze each neuron´s preference for direction and frequency we need to look at the signal directly after a stimulus presentation. Therefore, we split the data into segements for each trial.

In [ ]:
spike_times_all[0].shape

In [ ]:
spikes0 = spike_times_all[0]

In [ ]:
stims = data["stim_table"]
stims["start"] = stims["start"].astype(int)
stims["end"] = stims["end"].astype(int)
stims

In [ ]:
# flatten spikes into one row per spike
spike_rows = []

for neuron_idx, spike_times in enumerate(spike_times_all):
    for st in spike_times:
        spike_rows.append({"Neuron": neuron_idx, "SpikeTimes": st})

# create dataframe
spikes = pd.DataFrame(spike_rows)

# add new columns
spikes["Dir"] = np.nan
spikes["relTime"] = np.nan
spikes["Trial"] = np.nan
spikes["stimPeriod"] = np.nan

# determine unique directions
dirs = np.unique(stims["direction"])

# create trial counter for each direction
trialcounter = {d: 0 for d in dirs}

# iterate over stimulus trials
for i, row in stims.iterrows():
    # exclude nan values
    if pd.isna(row["direction"]) or pd.isna(row["start"]) or pd.isna(row["end"]):
        continue
    dir_now = row["direction"]
    blank_sweep = row["blank_sweep"]
    start = row["start"]
    end = row["end"]

    # increment trial counter
    trialcounter[dir_now] += 1

    # find spikes within trial window
    i0 = spikes["SpikeTimes"] > start
    i1 = spikes["SpikeTimes"] < end
    select = i0.values & i1.values

    spikes.loc[select, "Dir"] = dir_now
    spikes.loc[select, "Trial"] = trialcounter[dir_now]
    spikes.loc[select, "relTime"] = spikes.loc[select, "SpikeTimes"] - start
    spikes.loc[select, "stimPeriod"] = True

# keep only spikes inside any stim period
spikes = spikes.dropna(subset=["Dir"])
spikes

In [ ]:
spikes

In [ ]:
stimDur = 60  # duration of the stimulus in ms
deltaDir = 22.5  # direction step in degrees


def plotRaster(spikes: pd.DataFrame, neurons: list):
    """Plot spike rasters for multiple neurons sorted by condition.

    Parameters
    ----------
    spikes: pd.DataFrame
        Pandas DataFrame with columns
            Neuron | SpikeTimes | Dir | relTime | Trial | stimPeriod

    neurons: list
        List of neuron IDs to plot.

    Note
    ----
    This function does not return anything, it just creates a plot!
    """

    # -------------------------------------------------
    # Write a raster plot function for the data (2 pts)
    # -------------------------------------------------

    # create subplots based on the number of neurons
    n_neurons = len(neurons)
    n_cols = int(np.ceil(np.sqrt(n_neurons)))
    n_rows = int(np.ceil(n_neurons / n_cols))
    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows), sharex=True
    )

    # flatten axes array and trim excess if needed
    axes = axes.flatten()

    # ensure axes is iterable even for a single neuron
    if n_neurons == 1:
        axes = [axes]

    for ax, neuron in zip(axes, neurons):
        # filter data for the current neuron
        x = spikes.loc[spikes["Neuron"] == neuron, "relTime"]
        y = spikes.loc[spikes["Neuron"] == neuron, "Dir"]

        # create a scatter plot on the provided axis
        ax.scatter(x, y, marker="|", color="black", s=100)
        ax.set_xlabel("Time (ms)")
        ax.set_ylabel("Direction (deg)")
        ax.set_title(f"Neuron {neuron}")
        ax.set_yticks(np.arange(0, 360, deltaDir))
        ax.set_xticks(np.arange(0, stimDur + 1, 250))

    plt.tight_layout()
    plt.show()


plotRaster(spikes, neurons=list(range(1, 100)))  # Example for first four neurons

In [ ]:
# split data for one example neuron into stimulus epochs
def segment_data_by_stimulus(data, neuron=0, threshold=5):
    """
    Segment the dff data for a given neuron into stimulus epochs.

    Parameters:
    - data: The data dictionary containing dff data and stimulus table.
    - neuron: The index of the neuron to segment.

    Returns:
    - A list of segmented dff data for each stimulus epoch.
    """
    dff = data["dff"][neuron]
    stim_table = data["stim_table"]

    segments = []
    for _, row in stim_table.iterrows():
        start, end = int(row["start"]), int(row["end"])
        segments.append(dff[start:end])

    return segments


# example segmentation for one neuron
segmented_data = segment_data_by_stimulus(data, neuron=1)
# filter segemented data
filtered_segmented_data = [
    butterworth_filter(seg, cutoff=0.1, fs=1.0, order=5) for seg in segmented_data
]
# detect spikes in the segmented data for the first stimulus epoch
spikes_epoch_0, spike_times_epoch_0, threshold_epoch_0 = detect_spikes(
    filtered_segmented_data[0], fs=1000, N=threshold, lockout=1.0
)
# plot the segmented data for the first stimulus epoch
plt.figure(figsize=(10, 5))
plt.plot(filtered_segmented_data[0], label="Stimulus Epoch 1", color="blue", alpha=0.5)
plt.scatter(
    spikes_epoch_0,
    filtered_segmented_data[0][spikes_epoch_0],
    color="red",
    label="Detected Spikes",
    marker="x",
)
plt.title("Segmented DFF Data for Neuron 0 - Stimulus Epoch 1")
plt.xlabel("Time")
plt.ylabel("DFF")
plt.legend()
plt.show()

The segmentation for the first stimulus epoch worked, so we can apply it to all neurons. However, we can see no spike in this epoch. Therefore we then select the segments that have spikes to only extract tuning curves from them later.

In [ ]:
# plot example segment with spikes from the spikes DataFrame
plt.figure(figsize=(10, 5))

# Select the first row of the spikes DataFrame
neuron_idx = int(spikes.iloc[0]["Neuron"])
trial_idx = int(spikes.iloc[0]["Trial"])
stim = stims.iloc[trial_idx - 1]  # Get the corresponding stimulus trial
start, end = int(stim["start"]), int(stim["end"])

# Extract the segment for the selected neuron and trial
segment = data["dff"][neuron_idx][start:end]

# Detect spikes in the segment
spikes_in_segment, spike_times_in_segment, _ = detect_spikes(
    segment, fs=1000, N=threshold, lockout=1.0
)

# Plot the segment and detected spikes
plt.plot(segment, label="Segment with Spikes", color="blue", alpha=0.5)
plt.scatter(
    spikes_in_segment,
    segment[spikes_in_segment],
    color="red",
    label="Detected Spikes",
    marker="x",
)
plt.title(f"Segmented DFF Data for Neuron {neuron_idx} - Trial {trial_idx}")
plt.xlabel("Time")
plt.ylabel("DFF")
plt.legend()
plt.show()

Now we will correct each stimulus reaction spike by the baseline for which we use the activity in the timewindow of 10 ms before.

## Tuning functions

In [ ]:
# vonMises tuning curve estimation
def vonMises(
    theta: np.ndarray, alpha: float, kappa: float, ny: float, phi: float
) -> np.ndarray:
    """Evaluate the parametric von Mises tuning curve with parameters p at locations theta.

    Parameters
    ----------

    θ: np.array, shape=(N, )
        Locations. The input unit is degree.

    α, κ, ν, ϕ : float
        Function parameters

    Return
    ------
    f: np.array, shape=(N, )
        Tuning curve.
    """

    theta, phi = np.deg2rad(theta), np.deg2rad(phi)

    return np.exp(
        alpha + kappa * (np.cos(2 * (theta - phi)) - 1) + ny * (np.cos(theta - phi) - 1)
    )

In [ ]:
import scipy.optimize as opt


def tuningCurve(
    counts: np.ndarray, dirs: np.ndarray, neuron, show: bool = True
) -> np.ndarray:
    """Fit a von Mises tuning curve to the spike counts in count with direction dir using a least-squares fit.

    Parameters
    ----------

    counts: np.array, shape=(total_n_trials, )
        the spike count during the stimulation period

    dirs: np.array, shape=(total_n_trials, )
        the stimulus direction in degrees

    show: bool, default=True
        Plot or not.


    Return
    ------
    p: np.array or list, (4,)
        parameter vector of tuning curve function
    """
    (alpha, kappa, ny, phi), _ = opt.curve_fit(vonMises, dirs, counts, maxfev=100000)

    if show:

        theta = np.linspace(0, 360, 1000)
        fig, ax = plt.subplots(figsize=(8, 4))
        fig.suptitle(f"Von Mises Tuning Curve for Neuron {neuron}", fontsize=16)
        ax.plot(dirs, counts.mean(axis=1), "o", label="Data")
        ax.plot(theta, vonMises(theta, alpha, kappa, ny, phi), label="Fitted Curve")
        ax.set_xlabel("Direction (degrees)")
        ax.set_ylabel("Spike Count")
        ax.set_title(
            f"Alpha: {alpha:.2f}, Kappa: {kappa:.2f}, Ny: {ny:.2f}, Phi: {phi:.2f}"
        )
        ax.set_xlim(0, 360)
        ax.set_ylim(0, counts.max() * 1.2)
        ax.set_xticks(np.arange(0, 361, 45))
        ax.set_xticklabels(np.arange(0, 361, 45))
        ax.set_yticks(np.arange(0, counts.max() * 1.2, 5))
        ax.set_yticklabels(np.arange(0, counts.max() * 1.2, 5))
        ax.grid()
        ax.legend()
        plt.show()
    return np.array([alpha, kappa, ny, phi])

In [ ]:
spikes

In [ ]:
def get_data(neurons_df, neuron):
    # Filter for the given neuron
    spk_by_dir = (
        neurons_df[neurons_df["Neuron"] == neuron]
        .groupby(["Dir", "Trial"])
        .size()  # Count the number of spikes per direction and trial
        .reset_index(name="spike_count")
    )

    dirs = spk_by_dir["Dir"].values
    counts = spk_by_dir["spike_count"].values

    # Ensure zero entries for missing directions
    for direction in np.unique(neurons_df["Dir"]):
        if direction not in dirs:
            dirs = np.append(dirs, direction)
            counts = np.append(counts, 0)

    idx = np.argsort(dirs)
    dirs_sorted = dirs[idx]
    counts_sorted = counts[idx]

    return dirs_sorted, counts_sorted


# List of neurons to plot
neurons = [10, 51, 91, 97]  # Example neuron indices, adjust as needed

fig, axes = plt.subplots(len(neurons) // 2, 2, figsize=(12, 8), sharex=True)
fig.suptitle("Tuning Curves for Different Neurons", fontsize=16)
axes = axes.flatten()

for i, neuron in enumerate(neurons):
    ax = axes[i]
    dirs, counts = get_data(spikes, neuron)

    unique_dirs = np.sort(np.unique(dirs))
    df = pd.DataFrame({"Counts": counts, "Direction": dirs}).groupby("Direction").mean()

    α0 = np.log(np.max(df.values.flatten()) + 1e-3)
    ϕ0 = unique_dirs[np.argmax(df.values.flatten())]
    p0 = [α0, 1.0, 1.0, ϕ0]

    try:
        p, _ = opt.curve_fit(
            vonMises, df.index.values, df.values.flatten(), p0=p0, maxfev=10000
        )
    except RuntimeError:
        p = p0  # fallback to initial guess if fitting fails

    ax.plot(dirs, counts, "o", label="Data")
    ax.plot(df.index.values, df.values.flatten(), "o", label="Mean Spike Count")
    theta = np.linspace(0, 360, 1000)
    ax.plot(theta, vonMises(theta, *p), label="Fitted Curve")
    ax.set_title(f"Neuron {neuron}")
    ax.set_xlabel("Direction (degrees)")
    ax.set_ylabel("Spike Count")
    ax.set_xlim(0, 360)
    ax.set_ylim(0, counts.max() * 1.2)
    ax.set_xticks(np.arange(0, 361, 45))
    ax.grid()
    ax.legend()

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()